# Barrier Options — Further Topics

This is the follow-up to **06a (Barrier Options)**.  That notebook
covered the basics across all four pricing methods on well-behaved
contracts — barriers safely away from spot, contracts the engines all
converge on cleanly.  Here we cover the harder regimes that were flagged
but deferred:

1. **Engine speed** — the production engine (PDE_FD) is highly accurate
   by default (sub-bp on continuous European), with a speed mode available
   when throughput matters. Pricing continuous European barriers with default
   params generally takes a few seconds per valuation; when extra throughput
   is needed, switch to an explicit FD scheme (e.g. EXPLICIT_HULL) — 
   usually drops runtime to sub-second at the cost of a few bps of
   accuracy. For Americans
   an explicit scheme also requires american_solver=INTRINSIC, since
   GAUSS_SEIDEL (PSOR) isn't compatible with explicit time stepping.
   Discrete-monitoring barriers already default to this fast configuration
   (an explicit scheme at high time-step density outperforms implicit/CN
   at standard grids on both speed and accuracy).
2. **Truncated-payoff KO Americans** (DOP, UOC) — the worst-behaved
   barrier American specs, where MC LSM develops noticeable downward bias
   even with a barrier-aware basis.
3. **Barrier extremely close to spot** — using the Boyle-Tian (1998)
   Table 6 reference scenarios to demonstrate how the engines hold up as
   the barrier-to-spot distance shrinks.  Includes the Boyle-Lau step-cap
   on the binomial engine and the `_stencil_straddles_barrier` guard on
   binomial near-barrier greeks.
4. **Inception-triggered barriers** — specs where the barrier is
   already on the wrong side of spot at $t=0$.  Each engine recognises
   this natively and returns the deterministic value (KO → cash;
   KI → vanilla equivalent) for both PV and engine-native greeks.
   The OV (dispatcher) layer carries a separate short-circuit that
   fires for NUMERICAL bump-and-reprice greeks, where a naïve
   central difference would cross the trigger boundary.
5. **Redundant barriers** — KI specs whose barrier is automatically
   breached on any ITM path (UI call with $S_0 < H < K$, DI put with
   $S_0 > H > K$).  These price as vanilla without any short-circuit —
   a useful sanity check that the engines handle the limit correctly.
6. **Rebates — worked examples** showing how `AT_HIT` and `AT_EXPIRY`
   KO rebates and KI no-touch rebates change the PV.

## 1) Notebook Setup

In [1]:
from dataclasses import replace as dc_replace
import datetime as dt
import logging
import time
import math
import warnings

import derivatives_pricing as dp

# Silence the package's INFO/WARNING logger output for tidy tables
# below.  In §4.4 we re-enable briefly to demonstrate the spot-bump
# cap message.
logging.getLogger("derivatives_pricing").setLevel(logging.ERROR)

In [2]:
# ── Market / contract parameters (shared across most sections) ────────
pricing_date = dt.datetime(2025, 1, 1)
maturity = dt.datetime(2026, 1, 1)

S0 = 100.0
K = 100.0
sigma = 0.25
r = 0.05
q = 0.02

curve_r = dp.DiscountCurve.flat(r, end_time=2.0)
curve_q = dp.DiscountCurve.flat(q, end_time=2.0)

market_data = dp.MarketData(pricing_date, curve_r, currency="USD")
underlying = dp.UnderlyingData(
    initial_value=S0,
    volatility=sigma,
    market_data=market_data,
    dividend_curve=curve_q,
)

# GBM process used by Monte Carlo throughout.  We use 252 time steps
# rather than weekly (52) to reduce sampling-time discretisation
# error on the American LSM examples below.
MC_PATHS = 150_000
sim_config = dp.SimulationConfig(paths=MC_PATHS, end_date=maturity, num_steps=252)
gbm = dp.GBMProcess(
    market_data=market_data,
    process_params=dp.GBMParams(initial_value=S0, volatility=sigma, dividend_curve=curve_q),
    sim_config=sim_config,
)


def _safe(ov, accessor):
    """Call an OV accessor; return NaN if the call raises."""
    try:
        return getattr(ov, accessor)()
    except dp.DerivativesPricingError:
        return float("nan")


def _fmt(x, decimals=6):
    """Format a number, showing 'n/a' for NaN."""
    return "n/a" if (isinstance(x, float) and math.isnan(x)) else f"{x:.{decimals}f}"

## 2) Engine Speed: PDE_FD as the Production Engine

06b takes longer than notebook 06a because we exercise the slower
comparison engines:

- **Monte Carlo LSM** with 150k paths × 252 time steps (a few seconds
  per price).
- **Binomial with `num_steps=10_000`** for the converged-Binomial demos
  in §3.
- The default **Crank-Nicolson + projected SOR** PDE_FD for
  continuous-monitoring Americans (a few seconds per contract).

In day-to-day use you'd call `PDE_FD` with default parameters — fast
enough for most workflows.  When throughput matters, switch to an
explicit FD method.

In [3]:
# ── Speed mode for CONTINUOUS monitoring: EXPLICIT_HULL + INTRINSIC ────
# Drop CN+PSOR for EXPLICIT_HULL + INTRINSIC.  EXPLICIT_HULL's per-step
# cost is a vectorised matrix-vector multiply (no inner linear solve),
# so you can crank `time_steps` up cheaply.  Pass `fast_params` as the
# 4th argument to any `OptionValuation(..., PricingMethod.PDE_FD, ...)`
# call to use it.
fast_params = dp.PDEParams.for_barriers(
    monitoring=dp.BarrierMonitoring.CONTINUOUS,
    method=dp.PDEMethod.EXPLICIT_HULL,
    american_solver=dp.PDEEarlyExercise.INTRINSIC,
)
# For slightly better accuracy than the EXPLICIT_HULL defaults, bump
# `time_steps` — the matvec-per-step cost keeps wall time low even at
# 3-5x the default time-step count.

In [4]:
# ── Speed comparison: CN default vs EXPLICIT_HULL + INTRINSIC ──────────
def _bspec(option_type, exercise, direction, action, barrier):
    return dp.BarrierSpec(
        option_type=option_type,
        exercise_type=exercise,
        strike=K,
        maturity=maturity,
        barrier=barrier,
        direction=direction,
        action=action,
        monitoring=dp.BarrierMonitoring.CONTINUOUS,
    )


def _time_pde(spec, params=None):
    t0 = time.perf_counter()
    pv = dp.OptionValuation(underlying, spec, dp.PricingMethod.PDE_FD, params).present_value()
    return pv, time.perf_counter() - t0


def _bsm_pv_or_none(spec):
    """Analytical PV (Reiner-Rubinstein) for European; None for American."""
    if spec.exercise_type is not dp.ExerciseType.EUROPEAN:
        return None
    return float(dp.OptionValuation(underlying, spec, dp.PricingMethod.BSM).present_value())


speed_cases = [
    (
        "DOP european",
        _bspec(
            dp.OptionType.PUT,
            dp.ExerciseType.EUROPEAN,
            dp.BarrierDirection.DOWN,
            dp.BarrierAction.OUT,
            85.0,
        ),
    ),
    (
        "DOP american",
        _bspec(
            dp.OptionType.PUT,
            dp.ExerciseType.AMERICAN,
            dp.BarrierDirection.DOWN,
            dp.BarrierAction.OUT,
            85.0,
        ),
    ),
    (
        "UOC european",
        _bspec(
            dp.OptionType.CALL,
            dp.ExerciseType.EUROPEAN,
            dp.BarrierDirection.UP,
            dp.BarrierAction.OUT,
            115.0,
        ),
    ),
    (
        "UOC american",
        _bspec(
            dp.OptionType.CALL,
            dp.ExerciseType.AMERICAN,
            dp.BarrierDirection.UP,
            dp.BarrierAction.OUT,
            115.0,
        ),
    ),
    (
        "DIP american",
        _bspec(
            dp.OptionType.PUT,
            dp.ExerciseType.AMERICAN,
            dp.BarrierDirection.DOWN,
            dp.BarrierAction.IN,
            85.0,
        ),
    ),
    (
        "DIC european",
        _bspec(
            dp.OptionType.CALL,
            dp.ExerciseType.EUROPEAN,
            dp.BarrierDirection.DOWN,
            dp.BarrierAction.IN,
            85.0,
        ),
    ),
]

print("PDE_FD speed: CN default (PSOR) vs EXPLICIT_HULL + INTRINSIC")
print(
    f"{'case':<14} {'BSM PV':>10} {'CN PV':>10} {'CN time':>10} {'EH PV':>10} {'EH time':>10} "
    f"{'PV diff':>10} {'PV diff (%)':>14} {'speedup':>9}"
)
print("-" * 102)
for label, spec in speed_cases:
    bsm_pv = _bsm_pv_or_none(spec)
    cn_pv, cn_t = _time_pde(spec)
    eh_pv, eh_t = _time_pde(spec, fast_params)
    speedup = cn_t / eh_t if eh_t > 0 else float("inf")
    bsm_str = f"{bsm_pv:>10.4f}" if bsm_pv is not None else f"{'—':>10}"
    print(
        f"{label:<14} {bsm_str} {cn_pv:>10.4f} {cn_t * 1000:>8.0f}ms "
        f"{eh_pv:>10.4f} {eh_t * 1000:>8.0f}ms "
        f"{eh_pv - cn_pv:>+10.4f} "
        f"{((eh_pv - cn_pv) / cn_pv * 100 if cn_pv != 0 else float('inf')):>14.2f}% "
        f"{speedup:>8.1f}x"
    )

PDE_FD speed: CN default (PSOR) vs EXPLICIT_HULL + INTRINSIC
case               BSM PV      CN PV    CN time      EH PV    EH time    PV diff    PV diff (%)   speedup
------------------------------------------------------------------------------------------------------
DOP european       0.4167     0.4166      565ms     0.4187       73ms    +0.0021           0.50%      7.8x
DOP american            —     8.0129     1258ms     8.0346       81ms    +0.0217           0.27%     15.4x
UOC european       0.2623     0.2623      553ms     0.2635       73ms    +0.0011           0.44%      7.6x
UOC american            —     8.6920      958ms     8.7317       91ms    +0.0396           0.46%     10.5x
DIP american            —     8.1447      914ms     8.1477       86ms    +0.0031           0.04%     10.6x
DIC european       1.2003     1.2003     1107ms     1.1949      141ms    -0.0054          -0.45%      7.9x


**Discrete monitoring** already defaults to `EXPLICIT_HULL` +
`INTRINSIC` at `1200 × 3000` (see `PDEParams.for_barriers`), so
`dp.PDEParams.for_barriers(monitoring=BarrierMonitoring.DISCRETE)` is
already in fast mode — no extra configuration needed.

Hull's heuristic *targets* $dz_{\text{hull}} = \sigma\sqrt{3\,\Delta\tau}$
as the spatial step (which satisfies CFL). When
`spot_steps × dz_hull` covers the target log-spot range, $dz$ is set
to $dz_{\text{hull}}$ and stability is automatic.  If `time_steps` is
pushed high enough that Hull's grid would be too narrow, $dz$ is
rescaled up to `(zmax − zmin) / spot_steps` instead — still stable
(larger $dz$ relaxes CFL further), but coarser than Hull intended,
which can degrade spatial accuracy near the barrier.  Rule of thumb:
Hull's $dz_{\text{hull}}$ is binding when

$$\text{time\_steps} \;\lesssim\; 3\sigma^2 T \cdot \Bigl(\tfrac{\text{spot\_steps}}{\log(\text{smax\_mult}^2)}\Bigr)^2$$

For the DISCRETE defaults (σ=0.25, T=1, smax_mult=4, spot_steps=1200)
the ceiling is ~35,000 time steps — the 3000-step default has ~12×
headroom.  At that step density the reset discontinuities at each
observation date are typically damped over the next few steps.

The rest of this notebook uses CN defaults (continuous monitoring) so
the numbers compare directly to the academic references.

## 3) Truncated-Payoff KO Americans (DOP, UOC)

A **down-and-out put** (DOP) is killed by spot falling to the barrier $H$
— but the put's payoff $\max(K - S_T, 0)$ is *largest* exactly when spot
is low.  So the barrier truncates the contract in the part of the payoff
distribution that contributes most to value.  An **up-and-out call**
(UOC) is the symmetric story for the call.

Why this is hard for MC LSM:

- The American premium for these specs comes from *exercising just
  before the barrier kills the option*.  This is a sharp decision rule
  localised near $H$ — you want to exercise on paths heading toward the
  barrier and otherwise hold.
- LSM extracts that decision rule from a polynomial regression of
  continuation values onto realised paths.  Near the barrier, where
  the continuation surface has a near-discontinuity (continuation goes
  from "valuable alive option" to "killed, zero" as spot drops to $H$),
  the polynomial fit is noisy.  A noisy continuation estimate produces
  a sub-optimal exercise rule — and any sub-optimal rule on an American
  gives a **lower** estimated value than the true optimum (downward
  bias).
- Even with the package's `barrier_aware_basis` enrichment (extra
  regression features that explicitly model barrier-distance), the
  bias persists because the rare paths near the barrier are sparsely
  represented in any finite path sample.

  The package emits a warning at `__init__` for `MONTE_CARLO + AMERICAN +
  (DOP or UOC)` to flag this — see `monte_carlo.py:1649`.  The other
  barrier American specs (DI put, UI call, DO call, UI put, ...) don't
  trigger the warning; they converge cleanly.

In [5]:
# ── DOP setup ─────────────────────────────────────────────────────────
H_DOWN = 85.0

spec_dop_eu = dp.BarrierSpec(
    option_type=dp.OptionType.PUT,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=K,
    maturity=maturity,
    barrier=H_DOWN,
    direction=dp.BarrierDirection.DOWN,
    action=dp.BarrierAction.OUT,
    monitoring=dp.BarrierMonitoring.CONTINUOUS,
)
spec_dop_am = dc_replace(spec_dop_eu, exercise_type=dp.ExerciseType.AMERICAN)


def _price_ov(ud, spec, method):
    """Build OV and return PV; suppress LSM bias warning for tidy output."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return dp.OptionValuation(ud, spec, method).present_value()


bsm_dop_eu = _price_ov(underlying, spec_dop_eu, dp.PricingMethod.BSM)
binom_dop_eu = _price_ov(underlying, spec_dop_eu, dp.PricingMethod.BINOMIAL)
binom_dop_am = _price_ov(underlying, spec_dop_am, dp.PricingMethod.BINOMIAL)
pde_dop_eu = _price_ov(underlying, spec_dop_eu, dp.PricingMethod.PDE_FD)
pde_dop_am = _price_ov(underlying, spec_dop_am, dp.PricingMethod.PDE_FD)
mc_dop_eu = _price_ov(gbm, spec_dop_eu, dp.PricingMethod.MONTE_CARLO)
mc_dop_am = _price_ov(gbm, spec_dop_am, dp.PricingMethod.MONTE_CARLO)

# Take PDE_FD American as the reference to measure each engine's bias against.
ref_dop_am = pde_dop_am

print("── DOP (down-and-out put), barrier=85, strike=100 ──")
print(
    f"{'Engine':<14} {'European':>12} {'American':>12} {'EE prem':>12} "
    f"{'diff vs FD-AM':>16} {'diff (%)':>10}"
)
print("-" * 80)
print(f"{'BSM analytical':<14} {bsm_dop_eu:>12.6f} {'n/a':>12} {'n/a':>12} {'n/a':>16} {'n/a':>10}")
print(
    f"{'Binomial':<14} {binom_dop_eu:>12.6f} {binom_dop_am:>12.6f} "
    f"{binom_dop_am - binom_dop_eu:>+12.6f} {binom_dop_am - ref_dop_am:>+16.6f} "
    f"{(binom_dop_am - ref_dop_am) / ref_dop_am * 100:>+9.2f}%"
)
print(
    f"{'PDE_FD':<14} {pde_dop_eu:>12.6f} {pde_dop_am:>12.6f} "
    f"{pde_dop_am - pde_dop_eu:>+12.6f} {pde_dop_am - ref_dop_am:>+16.6f} "
    f"{(pde_dop_am - ref_dop_am) / ref_dop_am * 100:>+9.2f}%"
)
print(
    f"{'Monte Carlo':<14} {mc_dop_eu:>12.6f} {mc_dop_am:>12.6f} "
    f"{mc_dop_am - mc_dop_eu:>+12.6f} {mc_dop_am - ref_dop_am:>+16.6f} "
    f"{(mc_dop_am - ref_dop_am) / ref_dop_am * 100:>+9.2f}%"
)

── DOP (down-and-out put), barrier=85, strike=100 ──
Engine             European     American      EE prem    diff vs FD-AM   diff (%)
--------------------------------------------------------------------------------
BSM analytical     0.416656          n/a          n/a              n/a        n/a
Binomial           0.415635     7.921001    +7.505366        -0.091877     -1.15%
PDE_FD             0.416640     8.012878    +7.596238        +0.000000     +0.00%
Monte Carlo        0.418460     7.447772    +7.029312        -0.565106     -7.05%


**Reading the table.**  The PDE_FD American is the reference.

- **Binomial** tracks PDE_FD fairly well — just over one percent
  diff on the American.  Note even Binomial can be a few percentage
  points off PDE_FD on these truncated-payoff KO Americans (the same
  bias is observed in QuantLib's `BinomialCRRBarrierEngine`); Increasing
  `num_steps` to say 10,000 reduces the gap (see the next cell) at the
  cost of significantly increased runtime.
- **Monte Carlo's** American sits over 7% below the PDE_FD
  reference — a clear downward bias that path-count increase doesn't
  close.  The European prices agree to within MC sampling noise (the
  bias is exclusively on the American, where the LSM regression
  matters).

Below: Binomial DOP American with `num_steps=10_000` (vs the default
1000).  Notice how the gap to PDE_FD shrinks substantially.

In [6]:
binom_dop_am_10k = dp.OptionValuation(
    underlying,
    spec_dop_am,
    dp.PricingMethod.BINOMIAL,
    dp.BinomialParams(num_steps=10_000),
).present_value()
print(f"Binomial American DOP (num_steps=10_000): {binom_dop_am_10k:.6f}")
print(f"PDE_FD American DOP (reference):          {pde_dop_am:.6f}")
print(
    f"Gap: {binom_dop_am_10k - pde_dop_am:+.6f} "
    f"({(binom_dop_am_10k - pde_dop_am) / pde_dop_am * 100:+.2f}%)"
)

Binomial American DOP (num_steps=10_000): 7.996892
PDE_FD American DOP (reference):          8.012878
Gap: -0.015986 (-0.20%)


In [7]:
# ── UOC setup ─────────────────────────────────────────────────────────
H_UP = 115.0

spec_uoc_eu = dp.BarrierSpec(
    option_type=dp.OptionType.CALL,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=K,
    maturity=maturity,
    barrier=H_UP,
    direction=dp.BarrierDirection.UP,
    action=dp.BarrierAction.OUT,
    monitoring=dp.BarrierMonitoring.CONTINUOUS,
)
spec_uoc_am = dc_replace(spec_uoc_eu, exercise_type=dp.ExerciseType.AMERICAN)

bsm_uoc_eu = _price_ov(underlying, spec_uoc_eu, dp.PricingMethod.BSM)
binom_uoc_eu = _price_ov(underlying, spec_uoc_eu, dp.PricingMethod.BINOMIAL)
binom_uoc_am = _price_ov(underlying, spec_uoc_am, dp.PricingMethod.BINOMIAL)
pde_uoc_eu = _price_ov(underlying, spec_uoc_eu, dp.PricingMethod.PDE_FD)
pde_uoc_am = _price_ov(underlying, spec_uoc_am, dp.PricingMethod.PDE_FD)
mc_uoc_eu = _price_ov(gbm, spec_uoc_eu, dp.PricingMethod.MONTE_CARLO)
mc_uoc_am = _price_ov(gbm, spec_uoc_am, dp.PricingMethod.MONTE_CARLO)

ref_uoc_am = pde_uoc_am

print("\n── UOC (up-and-out call), barrier=115, strike=100 ──")
print(
    f"{'Engine':<14} {'European':>12} {'American':>12} {'EE prem':>12} "
    f"{'diff vs PDE-AM':>16} {'diff (%)':>10}"
)
print("-" * 80)
print(f"{'BSM analytical':<14} {bsm_uoc_eu:>12.6f} {'n/a':>12} {'n/a':>12} {'n/a':>16} {'n/a':>10}")
print(
    f"{'Binomial':<14} {binom_uoc_eu:>12.6f} {binom_uoc_am:>12.6f} "
    f"{binom_uoc_am - binom_uoc_eu:>+12.6f} {binom_uoc_am - ref_uoc_am:>+16.6f} "
    f"{(binom_uoc_am - ref_uoc_am) / ref_uoc_am * 100:>+9.2f}%"
)
print(
    f"{'PDE_FD':<14} {pde_uoc_eu:>12.6f} {pde_uoc_am:>12.6f} "
    f"{pde_uoc_am - pde_uoc_eu:>+12.6f} {pde_uoc_am - ref_uoc_am:>+16.6f} "
    f"{(pde_uoc_am - ref_uoc_am) / ref_uoc_am * 100:>+9.2f}%"
)
print(
    f"{'Monte Carlo':<14} {mc_uoc_eu:>12.6f} {mc_uoc_am:>12.6f} "
    f"{mc_uoc_am - mc_uoc_eu:>+12.6f} {mc_uoc_am - ref_uoc_am:>+16.6f} "
    f"{(mc_uoc_am - ref_uoc_am) / ref_uoc_am * 100:>+9.2f}%"
)


── UOC (up-and-out call), barrier=115, strike=100 ──
Engine             European     American      EE prem   diff vs PDE-AM   diff (%)
--------------------------------------------------------------------------------
BSM analytical     0.262330          n/a          n/a              n/a        n/a
Binomial           0.259242     8.483703    +8.224461        -0.208323     -2.40%
PDE_FD             0.262316     8.692026    +8.429709        +0.000000     +0.00%
Monte Carlo        0.261114     7.666875    +7.405761        -1.025151    -11.79%


The UOC shows even more pronounced divergence. Binomial is further
from the PDE_FD result (now ~2.4%). Monte Carlo's LSM is signifcantly biased
down vs the FD reference.  

In both MC cases the bias is structural (not a path-count artefact 
you'd shrink with more paths).  Increasing `MC_PATHS` reduces 
the *sampling* error band but does not move the
centre of the LSM distribution back toward the FD truth.

In [8]:
# Again Binomial converges to PDE_FD as num_steps increases.
binom_uoc_am_10k = dp.OptionValuation(
    underlying,
    spec_uoc_am,
    dp.PricingMethod.BINOMIAL,
    dp.BinomialParams(num_steps=10_000),
).present_value()
print(f"Binomial American UOC (num_steps=10_000): {binom_uoc_am_10k:.6f}")
print(f"PDE_FD American UOC (reference):          {pde_uoc_am:.6f}")
print(
    f"Gap: {binom_uoc_am_10k - pde_uoc_am:+.6f} "
    f"({(binom_uoc_am_10k - pde_uoc_am) / pde_uoc_am * 100:+.2f}%)"
)

Binomial American UOC (num_steps=10_000): 8.653743
PDE_FD American UOC (reference):          8.692026
Gap: -0.038283 (-0.44%)



**Practical guidance.**

- For **DOP and UOC** specifically, prefer **PDE_FD** for any
  production pricing.  Binomial is fine as a cross-check; MC LSM is
  for diagnostics only.
- For other American KO specs (DOC, UOP) the truncation is on the
  "out-of-the-money" side of the payoff and the bias is much smaller.
  All three numerical engines converge cleanly there.
- For all American KI specs (DIP, UIC, DIC, UIP), the LSM bias is
  small because once activated the contract is just a vanilla American
  and the regression has plenty of in-the-money paths.

Below — same setup but a **DOC** (down-and-out call), where the
barrier truncates only the deep-OTM tail of the payoff.  Notice MC's
bias collapses to MC sampling noise.

In [9]:
# ── DOC: a non-truncated-payoff KO American where MC behaves cleanly ──
spec_doc_eu = dp.BarrierSpec(
    option_type=dp.OptionType.CALL,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=K,
    maturity=maturity,
    barrier=H_DOWN,  # 85, well below spot
    direction=dp.BarrierDirection.DOWN,
    action=dp.BarrierAction.OUT,
    monitoring=dp.BarrierMonitoring.CONTINUOUS,
)
spec_doc_am = dc_replace(spec_doc_eu, exercise_type=dp.ExerciseType.AMERICAN)

bsm_doc_eu = _price_ov(underlying, spec_doc_eu, dp.PricingMethod.BSM)
binom_doc_eu = _price_ov(underlying, spec_doc_eu, dp.PricingMethod.BINOMIAL)
binom_doc_am = _price_ov(underlying, spec_doc_am, dp.PricingMethod.BINOMIAL)
pde_doc_eu = _price_ov(underlying, spec_doc_eu, dp.PricingMethod.PDE_FD)
pde_doc_am = _price_ov(underlying, spec_doc_am, dp.PricingMethod.PDE_FD)
mc_doc_eu = _price_ov(gbm, spec_doc_eu, dp.PricingMethod.MONTE_CARLO)
mc_doc_am = _price_ov(gbm, spec_doc_am, dp.PricingMethod.MONTE_CARLO)

ref_doc_am = pde_doc_am

print("\n── DOC (down-and-out call, NON-truncated payoff), barrier=85, strike=100 ──")
print(
    f"{'Engine':<14} {'European':>12} {'American':>12} {'EE prem':>12} "
    f"{'diff vs PDE-AM':>16} {'diff (%)':>10}"
)
print("-" * 80)
print(f"{'BSM analytical':<14} {bsm_doc_eu:>12.6f} {'n/a':>12} {'n/a':>12} {'n/a':>16} {'n/a':>10}")
print(
    f"{'Binomial':<14} {binom_doc_eu:>12.6f} {binom_doc_am:>12.6f} "
    f"{binom_doc_am - binom_doc_eu:>+12.6f} {binom_doc_am - ref_doc_am:>+16.6f} "
    f"{(binom_doc_am - ref_doc_am) / ref_doc_am * 100:>+9.2f}%"
)
print(
    f"{'PDE_FD':<14} {pde_doc_eu:>12.6f} {pde_doc_am:>12.6f} "
    f"{pde_doc_am - pde_doc_eu:>+12.6f} {pde_doc_am - ref_doc_am:>+16.6f} "
    f"{(pde_doc_am - ref_doc_am) / ref_doc_am * 100:>+9.2f}%"
)
print(
    f"{'Monte Carlo':<14} {mc_doc_eu:>12.6f} {mc_doc_am:>12.6f} "
    f"{mc_doc_am - mc_doc_eu:>+12.6f} {mc_doc_am - ref_doc_am:>+16.6f} "
    f"{(mc_doc_am - ref_doc_am) / ref_doc_am * 100:>+9.2f}%"
)


── DOC (down-and-out call, NON-truncated payoff), barrier=85, strike=100 ──
Engine             European     American      EE prem   diff vs PDE-AM   diff (%)
--------------------------------------------------------------------------------
BSM analytical     9.923414          n/a          n/a              n/a        n/a
Binomial           9.926217     9.926220    +0.000003        +0.002823     +0.03%
PDE_FD             9.923395     9.923398    +0.000003        +0.000000     +0.00%
Monte Carlo        9.945108     9.954575    +0.009467        +0.031177     +0.31%


DOC American agrees across all four engines to within MC sampling
noise (sub-1% MC vs PDE_FD).  No structural bias because the barrier
kills the option only in the *deep OTM* region — paths that would have
been ITM are unaffected, so the early-exercise decision rule lives in
a region where LSM regression has plenty of ITM paths to fit on.

And a **UIC** (up-and-in call) example. Again MC's bias collapses to MC sampling noise.

In [10]:
# ── UOC setup ─────────────────────────────────────────────────────────
H_UP = 115.0

spec_uic_eu = dp.BarrierSpec(
    option_type=dp.OptionType.CALL,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=K,
    maturity=maturity,
    barrier=H_UP,
    direction=dp.BarrierDirection.UP,
    action=dp.BarrierAction.IN,
    monitoring=dp.BarrierMonitoring.CONTINUOUS,
)
spec_uic_am = dc_replace(spec_uic_eu, exercise_type=dp.ExerciseType.AMERICAN)

bsm_uic_eu = _price_ov(underlying, spec_uic_eu, dp.PricingMethod.BSM)
binom_uic_eu = _price_ov(underlying, spec_uic_eu, dp.PricingMethod.BINOMIAL)
binom_uic_am = _price_ov(underlying, spec_uic_am, dp.PricingMethod.BINOMIAL)
pde_uic_eu = _price_ov(underlying, spec_uic_eu, dp.PricingMethod.PDE_FD)
pde_uic_am = _price_ov(underlying, spec_uic_am, dp.PricingMethod.PDE_FD)
mc_uic_eu = _price_ov(gbm, spec_uic_eu, dp.PricingMethod.MONTE_CARLO)
mc_uic_am = _price_ov(gbm, spec_uic_am, dp.PricingMethod.MONTE_CARLO)

ref_uic_am = pde_uic_am

print("\n── UIC (up-and-in call), barrier=115, strike=100 ──")
print(
    f"{'Engine':<14} {'European':>12} {'American':>12} {'EE prem':>12} "
    f"{'diff vs PDE-AM':>16} {'diff (%)':>10}"
)
print("-" * 80)
print(f"{'BSM analytical':<14} {bsm_uic_eu:>12.6f} {'n/a':>12} {'n/a':>12} {'n/a':>16} {'n/a':>10}")
print(
    f"{'Binomial':<14} {binom_uic_eu:>12.6f} {binom_uic_am:>12.6f} "
    f"{binom_uic_am - binom_uic_eu:>+12.6f} {binom_uic_am - ref_uic_am:>+16.6f} "
    f"{(binom_uic_am - ref_uic_am) / ref_uic_am * 100:>+9.2f}%"
)
print(
    f"{'PDE_FD':<14} {pde_uic_eu:>12.6f} {pde_uic_am:>12.6f} "
    f"{pde_uic_am - pde_uic_eu:>+12.6f} {pde_uic_am - ref_uic_am:>+16.6f} "
    f"{(pde_uic_am - ref_uic_am) / ref_uic_am * 100:>+9.2f}%"
)
print(
    f"{'Monte Carlo':<14} {mc_uic_eu:>12.6f} {mc_uic_am:>12.6f} "
    f"{mc_uic_am - mc_uic_eu:>+12.6f} {mc_uic_am - ref_uic_am:>+16.6f} "
    f"{(mc_uic_am - ref_uic_am) / ref_uic_am * 100:>+9.2f}%"
)


── UIC (up-and-in call), barrier=115, strike=100 ──
Engine             European     American      EE prem   diff vs PDE-AM   diff (%)
--------------------------------------------------------------------------------
BSM analytical    10.861432          n/a          n/a              n/a        n/a
Binomial          10.862186    10.862189    +0.000003        +0.000634     +0.01%
PDE_FD            10.861345    10.861554    +0.000209        +0.000000     +0.00%
Monte Carlo       10.879491    10.753886    -0.125605        -0.107668     -0.99%


## 4) Barrier Extremely Close to Spot (Boyle-Tian Table 6)

Boyle & Tian (1998) report analytical PVs (Tables 1 & 4) and greeks (Table 6)
for a down-and-out call as spot is varied from 95 down to 90.2 with the
barrier fixed at 90.  The aim is to assess the accuracy of hedging
parameters in the spot-near-barrier cases, for their explicit FD approach.
This is exactly the regime where:

- **Binomial Boyle-Lau alignment** can fail: the formula
  $N_i = \lfloor i^2 \sigma^2 T / (\ln H/S)^2 \rfloor$ becomes huge as
  $\ln(H/S) \to 0$.  The package caps the tree at
  `max(1000, 5 × num_steps)` and emits a `RuntimeWarning` when that cap
  binds — at which point the tree runs without barrier alignment and
  only achieves $O(1/\sqrt{N})$ convergence.
- **Binomial near-barrier greeks** also have a separate guard,
  `_stencil_straddles_barrier` (`binomial.py:1053`).  Hull's tree
  delta uses a step-1 central difference (the up- and down-nodes
  from the root); the gamma uses the step-2 stencil.  When spot is
  close enough to the barrier that one of those nodes lies past $H$,
  the central-difference formulas read across the absorbing-boundary
  discontinuity as huge curvature.  The engine raises
  `UnsupportedFeatureError` rather than return a misleading value.
- **PDE_FD's truncated-grid approach** stays accurate: the barrier is
  the grid boundary regardless of how close it is to spot.
- **BSM analytical PV** is exact regardless of barrier-to-spot
  distance.  Note however that the `derivatives-pricing` package does 
  **not** currently implement closed-form analytical greeks for barriers.
  BSM barrier greeks resolve to NUMERICAL bump-and-revalue.  So our BSM 
  greeks are close to the paper's but don't match exactly.

Setup (from the paper):
- Strike $K = 100$, barrier $H = 90$, maturity $T = 1$ year.
- $\sigma = 0.25$, $r = 0.10$, $q = 0$.
- Spot varied across $\{95, 92, 91, 90.5, 90.4, 90.3, 90.2\}$.

In [11]:
# ── BT98 Table 6 setup ────────────────────────────────────────────────
# Paper-reported PV and (delta, gamma, theta_annualised) for each spot.
PAPER_PVS = {
    95.0: 5.9968,
    92.0: 2.5063,
    91.0: 1.2738,
    90.5: 0.6424,
    90.4: 0.5148,
    90.3: 0.3868,
    90.2: 0.2583,
}

PAPER_GREEKS = {
    # spot: (delta, gamma, theta_annualised)
    95.0: (1.1192, -0.0262, -2.6468),
    92.0: (1.2132, -0.0370, -1.1229),
    91.0: (1.2524, -0.0413, -0.5720),
    90.5: (1.2736, -0.0437, -0.2886),
    90.4: (1.2780, -0.0441, -0.2313),
    90.3: (1.2825, -0.0446, -0.1738),
    90.2: (1.2869, -0.0451, -0.1161),
}

paper_curve_r = dp.DiscountCurve.flat(0.10, 2.0)
paper_curve_q = dp.DiscountCurve.flat(0.0, 2.0)
paper_md = dp.MarketData(pricing_date, paper_curve_r, currency="USD")


def _paper_underlying(spot: float) -> dp.UnderlyingData:
    return dp.UnderlyingData(
        initial_value=spot,
        volatility=0.25,
        market_data=paper_md,
        dividend_curve=paper_curve_q,
    )


paper_spec = dp.BarrierSpec(
    option_type=dp.OptionType.CALL,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=100.0,
    maturity=maturity,
    barrier=90.0,
    direction=dp.BarrierDirection.DOWN,
    action=dp.BarrierAction.OUT,
    monitoring=dp.BarrierMonitoring.CONTINUOUS,
)

### 4.1 PV vs paper as spot → barrier

Paper-reported PVs are reproduced alongside the package's three engines.
BSM closed-form PV matches the paper to all displayed decimals
(it's the same Reiner-Rubinstein formula evaluated on the same
inputs).

In [12]:
# ── Build a per-(spot, engine) cache once: one OptionValuation gives
# PV + Δ + Γ + Θ via the engines' cached lattice solves.  §4.1 (this
# cell) prints PV from the cache; §4.2 prints greeks from the same
# cache — no recomputation. ────────────────────────────────────────────
GREEKS = ("delta", "gamma", "theta")
_QUANTITIES = ("pv", *GREEKS)


def _pv_and_greeks(spot: float, method: dp.PricingMethod) -> dict[str, tuple[float, bool]]:
    """PV + Δ + Γ + Θ for one (spot, method) from a single OptionValuation.

    Binomial and PDE_FD engines cache their lattice solve, so calling
    ``ov.present_value()``, ``ov.delta()``, ``ov.gamma()``, ``ov.theta()``
    on the same ``ov`` is **one solve + four cached reads** — vs.
    creating a fresh ``OptionValuation`` per quantity (the previous
    pattern), which paid for four full solves.

    Returns ``{key: (value, warned)}`` for ``key`` in
    ``{"pv", "delta", "gamma", "theta"}``.  NaN value means the engine
    raised for that quantity (e.g. Binomial's
    ``_stencil_straddles_barrier`` guard).  ``warned=True`` means the
    package emitted a Boyle-Lau cap warning during the solve.
    """
    ud = _paper_underlying(spot)
    result: dict[str, tuple[float, bool]] = {k: (float("nan"), False) for k in _QUANTITIES}
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", RuntimeWarning)
        try:
            ov = dp.OptionValuation(ud, paper_spec, method)
        except dp.DerivativesPricingError:
            return result
        for k in _QUANTITIES:
            fn_name = "present_value" if k == "pv" else k
            try:
                val = float(getattr(ov, fn_name)())
                if k == "theta":
                    val *= 365.0  # annualise to match paper convention
                result[k] = (val, False)
            except dp.DerivativesPricingError:
                pass  # leave as NaN; display layer renders "n/a"
    warned = any("Boyle-Lau step alignment" in str(w.message) for w in caught)
    if warned:
        result = {k: (v, True) if not math.isnan(v) else (v, False) for k, (v, _) in result.items()}
    return result


def _fmt_num(val_warned: tuple[float, bool], decimals: int = 4) -> str:
    """Format a greek; append ``*`` when the Boyle-Lau cap warned."""
    value, warned = val_warned
    if isinstance(value, float) and math.isnan(value):
        return "n/a"
    return f"{value:.{decimals}f}" + ("*" if warned else " ")


_engine_methods = (dp.PricingMethod.BSM, dp.PricingMethod.BINOMIAL, dp.PricingMethod.PDE_FD)
engine_cache: dict[tuple[float, dp.PricingMethod], dict[str, tuple[float, bool]]] = {
    (spot, method): _pv_and_greeks(spot, method)
    for spot in PAPER_GREEKS.keys()
    for method in _engine_methods
}

print("DOC PV across engines as spot → barrier (H = 90):")
print(f"{'spot':<6} {'paper':>10} {'BSM':>10} {'Binomial':>16} {'PDE_FD':>14}")
print("-" * 60)
for spot in PAPER_GREEKS.keys():
    bsm = engine_cache[(spot, dp.PricingMethod.BSM)]["pv"]
    binom = engine_cache[(spot, dp.PricingMethod.BINOMIAL)]["pv"]
    pde = engine_cache[(spot, dp.PricingMethod.PDE_FD)]["pv"]
    print(
        f"{spot:<6.1f} {PAPER_PVS[spot]:>10.4f} "
        f"{_fmt_num(bsm):>12} {_fmt_num(binom):>14} {_fmt_num(pde):>14}"
    )

print("\n* = Boyle-Lau cap warning fired (number still produced, at degraded O(1/√n) convergence).")

DOC PV across engines as spot → barrier (H = 90):
spot        paper        BSM         Binomial         PDE_FD
------------------------------------------------------------
95.0       5.9968      5.9968         5.9986         5.9968 
92.0       2.5063      2.5063         2.5069         2.5062 
91.0       1.2738      1.2738         1.2740         1.2738 
90.5       0.6424      0.6424         0.6424         0.6423 
90.4       0.5148      0.5148         0.5148         0.5147 
90.3       0.3868      0.3868         0.4105*        0.3867 
90.2       0.2583      0.2583         0.4086*        0.2582 

* = Boyle-Lau cap warning fired (number still produced, at degraded O(1/√n) convergence).


- **PDE_FD** matches BSM closely across all spots, including down to
  $S = 90.2$ where the barrier is only 0.2 away.
- **Binomial** matches at the wider spots but degrades to a still-priced
  but inaccurate value at $S \le 90.3$ — the Boyle-Lau cap binds and
  the tree falls back to running unaligned (you'll see the warning if
  you don't suppress it; we surface it in §4.3).

### 4.2 Greeks vs the paper's reference values

Now onto Boyle-Tian Table 6: greeks across the same spots.
Theta in the package is per-day; multiply by 365 to compare to the
paper's annualised values.

In [13]:
# Greeks are pulled from `engine_cache` (built once in §4.1) — no
# recomputation per greek.  See the cache-build cell for the per-engine
# semantics (NaN for raised, `*` for Boyle-Lau cap warning).
for greek in GREEKS:
    print(f"\n── {greek.upper()} (BT98 Table 6) ──")
    if greek == "theta":
        print("   (annualised; library theta × 365)")
    print(f"{'spot':<6} {'paper':>10} {'BSM':>10} {'Binomial':>12} {'PDE_FD':>12}")
    print("-" * 56)
    paper_idx = GREEKS.index(greek)
    for spot, paper_triple in PAPER_GREEKS.items():
        paper_val = paper_triple[paper_idx]
        bsm = engine_cache[(spot, dp.PricingMethod.BSM)][greek]
        binom = engine_cache[(spot, dp.PricingMethod.BINOMIAL)][greek]
        pde = engine_cache[(spot, dp.PricingMethod.PDE_FD)][greek]
        print(
            f"{spot:<6.1f} {paper_val:>10.4f} {_fmt_num(bsm):>10} "
            f"{_fmt_num(binom):>12} {_fmt_num(pde):>12}"
        )

print(
    "\n* = Boyle-Lau cap warning fired (number still produced, at "
    "degraded O(1/√n) convergence).  n/a = engine raised "
    "(`_stencil_straddles_barrier` for Binomial near-barrier greeks)."
)


── DELTA (BT98 Table 6) ──
spot        paper        BSM     Binomial       PDE_FD
--------------------------------------------------------
95.0       1.1192    1.1197       1.1189       1.1192 
92.0       1.2132    1.2138       1.2131       1.2132 
91.0       1.2524    1.2526       1.2522       1.2524 
90.5       1.2736    1.2737           n/a      1.2736 
90.4       1.2780    1.2780           n/a      1.2780 
90.3       1.2825    1.2825           n/a      1.2825 
90.2       1.2869    1.2869           n/a      1.2869 

── GAMMA (BT98 Table 6) ──
spot        paper        BSM     Binomial       PDE_FD
--------------------------------------------------------
95.0      -0.0262   -0.0262      -0.0262      -0.0262 
92.0      -0.0370   -0.0370      -0.0370      -0.0370 
91.0      -0.0413   -0.0413           n/a     -0.0413 
90.5      -0.0437   -0.0437           n/a     -0.0437 
90.4      -0.0441   -0.0441           n/a     -0.0441 
90.3      -0.0446   -0.0446           n/a     -0.0446 
90.2 

- **BSM** matches the paper closely on delta/gamma; the gap in theta
  is slightly greater, though still small.
- **PDE_FD** matches the paper very closely on all three greeks across the
  full spot range.
- **Binomial** behaves piece-wise:
  - At spots comfortably above the barrier (95, 92), all three greeks
    are very close to the paper values.
  - **Gamma `n/a` at spot 91 onwards**: `_stencil_straddles_barrier`
    fires for `step=2`.  At $S=91$ with the CRR tree, the dd-node
    from the root (two down-moves) lands below $H = 90$, so the
    three-node central-difference stencil for gamma reads across
    the absorbing-boundary discontinuity.  The engine raises
    `UnsupportedFeatureError` — better than a misleadingly large
    gamma.
  - **Delta `n/a` at spot 90.5 onwards**: same guard at `step=1`.
    At $S=90.5$ the down-node from the root lands below $H$ → the
    two-node delta stencil straddles the barrier → guard fires.
  - **Theta `*` at spot 90.3 onwards**: this one is from the
    **Boyle-Lau cap** (separate from the stencil guard).  At $S=90.3$
    the alignment formula wants ~5643 tree steps but the cap is at
    5000, so the engine emits `RuntimeWarning`. We put an asterisk
    after the number accordingly.

**Practical guidance.** when $|\ln(S/H)| < 0.01$ (barrier within ~1%
of spot, log-space), use PDE_FD for both PV and greeks.
Binomial is fine for less-extreme barrier proximity; BSM is exact on 
PV regardless of proximity but currently trades a few bps of theta
accuracy because we use NUMERICAL bumps for barrier analytical-engine greeks.

### 4.3 What the Binomial cap warning looks like

Let's price one of the close-barrier cases without suppressing
warnings so you can see what the package emits.

In [14]:
print("Pricing DOC at spot=90.3 with BINOMIAL (no warning suppression):")
ud = _paper_underlying(90.3)
# The Boyle-Lau cap warning fires at OV construction time (inside
# `_resolve_effective_num_steps`), so we instantiate inside the
# catch_warnings block to capture it.
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", RuntimeWarning)
    ov_binom = dp.OptionValuation(ud, paper_spec, dp.PricingMethod.BINOMIAL)
    pv = ov_binom.present_value()
print(f"  PV = {pv:.6f}  ({len(caught)} warning(s) emitted)")
for w in caught:
    print(f"  ⚠ {w.category.__name__}: {str(w.message)[:140]}…")

Pricing DOC at spot=90.3 with BINOMIAL (no warning suppression):
  PV = 0.410470  (1 warning(s) emitted)
  ⚠ RuntimeWarning: Binomial barrier pricing: Boyle-Lau step alignment needs ~5643 time steps to place a tree layer on the barrier (spot=90.3, barrier=90, |log(…


The warning tells you exactly what's happening: Boyle-Lau alignment
wants more tree steps than the cap allows.  The tree falls back to
running at the cap *without* alignment, so the price has unaligned
barrier-misplacement bias.  Switch engines.

### 4.4 Numerical greek bumps near the barrier

When you compute delta or gamma via NUMERICAL bump-and-revalue (e.g.
on the BSM engine, or on Monte Carlo), the central-difference scheme
bumps spot up and down by $\pm\varepsilon$ and re-prices.  If
$\varepsilon$ is too large the bumped spot crosses the barrier and you
end up pricing an entirely different contract (alive vs. dead).

The package automatically **caps** the spot bump on barrier specs so
the bumped spot stays on the same side of $H$ as $S_0$.  Specifically,
$\varepsilon \to \min(\varepsilon, 0.5 \cdot |S_0 - H|)$ — at most
halfway to the barrier.  This is in `OptionValuation._BARRIER_BUMP_MAX_FRACTION`
and is invoked in `delta()` / `gamma()` whenever the spec is a
`BarrierSpec`.

Below: BSM numerical delta on the close-barrier DOC, demonstrating
that even with an unnaturally large user-supplied $\varepsilon$ the
package still produces a sensible delta (because it caps internally).

In [15]:
ud_close = _paper_underlying(90.5)  # spot 0.5 above barrier
ov_bsm_close = dp.OptionValuation(ud_close, paper_spec, dp.PricingMethod.BSM)

# Capture the cap-clipping messages with a list handler so we can
# display them deterministically after the delta calls (the package
# emits them at WARNING level via the standard logging system).
captured_log_msgs: list[str] = []


class _ListHandler(logging.Handler):
    def emit(self, record):
        captured_log_msgs.append(record.getMessage())


_handler = _ListHandler()
_handler.setLevel(logging.WARNING)
_pkg_logger = logging.getLogger("derivatives_pricing")
_pkg_logger.addHandler(_handler)
_pkg_logger.setLevel(logging.WARNING)

# Auto bump (defaults to spot/100 ≈ 0.9, which exceeds 0.5·|S-H|=0.25 → cap binds).
delta_default = ov_bsm_close.delta()

# User passes a 5-point bump — way too big.  Cap should kick in.
delta_huge_eps = ov_bsm_close.delta(
    epsilon=5.0, greek_calc_method=dp.GreekCalculationMethod.NUMERICAL
)

# Restore silence for the rest of the notebook.
_pkg_logger.removeHandler(_handler)
_pkg_logger.setLevel(logging.ERROR)

paper_delta_905 = PAPER_GREEKS[90.5][0]
print(f"BSM numerical delta @ spot=90.5 (paper {paper_delta_905:.4f}):")
print(f"  default ε:           {delta_default:.4f}")
print(f"  user ε=5 (cap binds): {delta_huge_eps:.4f}")
print()
print("Cap-clipping log messages captured during the two delta calls:")
for msg in captured_log_msgs:
    print(f"  ⚠ {msg}")
print()
print(
    "Both deltas are close to the paper value because the cap shrinks "
    "the bump to stay on the alive side of the barrier."
)

BSM numerical delta @ spot=90.5 (paper 1.2736):
  default ε:           1.2737
  user ε=5 (cap binds): 1.2737

Cap-clipping log messages captured during the two delta calls:
  ⚠ Numerical greek bump epsilon=0.905 would cross DOWN barrier H=90 (spot=90.5); shrinking to 0.25.
  ⚠ Numerical greek bump epsilon=5 would cross DOWN barrier H=90 (spot=90.5); shrinking to 0.25.

Both deltas are close to the paper value because the cap shrinks the bump to stay on the alive side of the barrier.


## 5) Inception-Triggered Barriers

A barrier spec is **triggered at inception** when the spot at the
pricing date is already on the wrong side of the barrier (under
continuous monitoring) or, for discrete monitoring, the pricing date
is itself an observation date with spot on the wrong side.  Two
pricing regimes collapse to a deterministic answer:

- **Triggered KO**: the contract is extinguished.  PV is the
  deterministic rebate cashflow:
    - 0 if no rebate
    - $R$ if rebate is paid AT_HIT (already paid, sitting as cash)
    - $R \cdot \text{df}(0, T)$ if AT_EXPIRY
- **Triggered KI**: the contract is activated, so it *is* the
  underlying vanilla.  PV equals the vanilla equivalent's PV; greeks
  match the vanilla equivalent's greeks (same option type, strike,
  maturity, exercise style).

**Where the handling lives.**  PV and engine-native greeks
(TREE / GRID) are produced by each
engine *natively* — every engine recognises inception-triggered
specs and returns the deterministic value itself.  The OV
(dispatcher) layer in `core.py` carries a separate, narrowly-scoped
short-circuit for NUMERICAL bump-and-reprice greeks:
bumping spot may cross the trigger boundary and revalue the
un-triggered contract, so OV intercepts before the bump fires and
either returns the constant-cashflow derivative (KO) or delegates
to a cached `_vanilla_equivalent_valuation` (KI).

### 5.1 Detecting a triggered spec

`BarrierSpec.is_spot_past_barrier` is a pure-spec helper that answers "would
this barrier be considered hit at the given spot?" without needing
to construct an `OptionValuation`.

In [16]:
spec_down_85 = dp.BarrierSpec(
    option_type=dp.OptionType.CALL,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=100.0,
    maturity=maturity,
    barrier=85.0,
    direction=dp.BarrierDirection.DOWN,
    action=dp.BarrierAction.OUT,
    monitoring=dp.BarrierMonitoring.CONTINUOUS,
)
print("BarrierSpec.is_spot_past_barrier demo (DOWN barrier at H=85):")
for s in (100.0, 90.0, 85.0, 80.0):
    print(f"  spot={s:>6.1f}  is_spot_past_barrier = {spec_down_85.is_spot_past_barrier(s)}")

BarrierSpec.is_spot_past_barrier demo (DOWN barrier at H=85):
  spot= 100.0  is_spot_past_barrier = False
  spot=  90.0  is_spot_past_barrier = False
  spot=  85.0  is_spot_past_barrier = True
  spot=  80.0  is_spot_past_barrier = True


For a DOWN barrier, `is_spot_past_barrier(spot) = (spot <= barrier)` — the
spec records the trigger any time spot has reached or fallen below
the barrier.  Symmetrically for UP: `is_spot_past_barrier(spot) = (spot >= barrier)`.

### 5.2 Triggered KO — deterministic cash

Construct an underlying where spot starts at 80 and the barrier is at
85 (DOWN-OUT).  At inception the option is dead.

In [17]:
ud_triggered = dp.UnderlyingData(
    initial_value=80.0,
    volatility=sigma,
    market_data=market_data,
    dividend_curve=curve_q,
)

spec_ko_no_rebate = dp.BarrierSpec(
    option_type=dp.OptionType.CALL,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=100.0,
    maturity=maturity,
    barrier=85.0,
    direction=dp.BarrierDirection.DOWN,
    action=dp.BarrierAction.OUT,
    monitoring=dp.BarrierMonitoring.CONTINUOUS,
)
spec_ko_at_hit = dc_replace(spec_ko_no_rebate, rebate=5.0, rebate_timing=dp.RebateTiming.AT_HIT)
spec_ko_at_expiry = dc_replace(
    spec_ko_no_rebate, rebate=5.0, rebate_timing=dp.RebateTiming.AT_EXPIRY
)

# Same trio of triggered KO contracts priced with all four engines.
print("Triggered DOWN-and-out call (spot=80, barrier=85, K=100):")
print(f"{'Rebate':<22} {'BSM':>10} {'Binomial':>12} {'PDE_FD':>12} {'MC':>12}")
print("-" * 70)
for label, spec in [
    ("none (R=0)", spec_ko_no_rebate),
    ("AT_HIT (R=5)", spec_ko_at_hit),
    ("AT_EXPIRY (R=5)", spec_ko_at_expiry),
]:
    pvs = []
    for method, src in (
        (dp.PricingMethod.BSM, ud_triggered),
        (dp.PricingMethod.BINOMIAL, ud_triggered),
        (dp.PricingMethod.PDE_FD, ud_triggered),
        (
            dp.PricingMethod.MONTE_CARLO,
            dp.GBMProcess(
                market_data=market_data,
                process_params=dp.GBMParams(
                    initial_value=80.0, volatility=sigma, dividend_curve=curve_q
                ),
                sim_config=sim_config,
            ),
        ),
    ):
        pvs.append(dp.OptionValuation(src, spec, method).present_value())
    print(f"{label:<22} {pvs[0]:>10.6f} {pvs[1]:>12.6f} {pvs[2]:>12.6f} {pvs[3]:>12.6f}")

Triggered DOWN-and-out call (spot=80, barrier=85, K=100):
Rebate                        BSM     Binomial       PDE_FD           MC
----------------------------------------------------------------------
none (R=0)               0.000000     0.000000     0.000000     0.000000
AT_HIT (R=5)             5.000000     5.000000     5.000000     5.000000
AT_EXPIRY (R=5)          4.756147     4.756147     4.756147     4.756147


**Reading the table.**

- All four engines return **identical** PVs to many decimal places —
  each engine independently detects the inception-triggered state
  and returns the deterministic cashflow value.
- **No rebate**: PV = 0 (option dead, nothing payable).
- **AT_HIT rebate=5**: PV = 5.0 — the rebate is "paid" at the trigger
  moment, which at inception is *now*, so it sits as cash with no
  discounting.
- **AT_EXPIRY rebate=5**: PV = $5 \cdot e^{-rT}$ — the rebate will be
  paid at maturity, so we discount to today.

### 5.3 Triggered KO — greeks

Greeks of a triggered KO follow directly from the cashflow being
deterministic in spot and (for AT_HIT) in time:

In [18]:
ov_ko = dp.OptionValuation(ud_triggered, spec_ko_at_expiry, dp.PricingMethod.BSM)

print("Triggered DOWN-and-out call greeks (AT_EXPIRY rebate R=5):")
print(f"  PV    = {ov_ko.present_value():.6f}            (= 5 · df(0, T))")
print(f"  Δ     = {ov_ko.delta():.6f}            (cashflow constant in spot)")
print(f"  Γ     = {ov_ko.gamma():.6f}            (constant in spot)")
print(f"  vega  = {ov_ko.vega():.6f}            (vol-insensitive cashflow)")
print(f"  Θ/day = {ov_ko.theta():.6f}            (= r · PV / 365 — the rebate accrues)")
print(f"  ρ     = {ov_ko.rho():.6f}            (sensitive to rate via discount factor)")

Triggered DOWN-and-out call greeks (AT_EXPIRY rebate R=5):
  PV    = 4.756147            (= 5 · df(0, T))
  Δ     = 0.000000            (cashflow constant in spot)
  Γ     = 0.000000            (constant in spot)
  vega  = 0.000000            (vol-insensitive cashflow)
  Θ/day = 0.000652            (= r · PV / 365 — the rebate accrues)
  ρ     = -0.047562            (sensitive to rate via discount factor)


All five greeks have closed-form values regardless of engine selection.
Notable: theta is **not** zero for the AT_EXPIRY rebate (the discount
factor unwinds as time passes, so the cash payment grows in present
value at rate $r$).  Theta for an AT_HIT or no-rebate triggered KO
*is* zero (the cashflow is constant in time).

### 5.4 Triggered KI — collapses to vanilla

Symmetric story for KI: at inception the option is already activated,
so it equals the underlying vanilla.

In [19]:
spec_ki = dp.BarrierSpec(
    option_type=dp.OptionType.CALL,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=100.0,
    maturity=maturity,
    barrier=85.0,
    direction=dp.BarrierDirection.DOWN,
    action=dp.BarrierAction.IN,
    monitoring=dp.BarrierMonitoring.CONTINUOUS,
)

spec_vanilla = dp.VanillaSpec(
    option_type=dp.OptionType.CALL,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=100.0,
    maturity=maturity,
)

ov_ki_bsm = dp.OptionValuation(ud_triggered, spec_ki, dp.PricingMethod.BSM)
ov_van = dp.OptionValuation(ud_triggered, spec_vanilla, dp.PricingMethod.BSM)

# Force NUMERICAL on both sides so the comparison is apples-to-apples.
# When the caller requests NUMERICAL greeks on a triggered KI, OV
# routes the call through vanilla_equivalent.<greek>(NUMERICAL); the
# externally-built vanilla with the same NUMERICAL request then
# bump-and-revalues with the same epsilon and produces byte-identical
# numbers.  (For engine-native greeks — ANALYTICAL / TREE / GRID /
# PATHWISE / LR — the engine handles the triggered KI itself, with no
# OV routing involved.)
NUM = dp.GreekCalculationMethod.NUMERICAL

LABEL_W = 16
VAL_W = 14

print("Triggered DOWN-and-in call (spot=80, barrier=85, K=100) vs vanilla call:")
print()
print(f"{'':<{LABEL_W}} {'KI triggered':>{VAL_W}} {'Vanilla':>{VAL_W}} {'diff':>{VAL_W}}")
print("-" * (LABEL_W + 3 * (VAL_W + 1)))

for accessor, kwargs in [
    ("present_value", {}),
    ("delta", {"greek_calc_method": NUM}),
    ("gamma", {"greek_calc_method": NUM}),
    ("vega", {"greek_calc_method": NUM}),
    ("theta", {"greek_calc_method": NUM}),
    ("rho", {"greek_calc_method": NUM}),
]:
    try:
        ki_val = float(getattr(ov_ki_bsm, accessor)(**kwargs))
    except dp.DerivativesPricingError:
        ki_val = float("nan")

    try:
        van_val = float(getattr(ov_van, accessor)(**kwargs))
    except dp.DerivativesPricingError:
        van_val = float("nan")

    print(
        f"{accessor:<{LABEL_W}} "
        f"{_fmt(ki_val, 6):>{VAL_W}} "
        f"{_fmt(van_val, 6):>{VAL_W}} "
        f"{_fmt(ki_val - van_val, 6):>{VAL_W}}"
    )

Triggered DOWN-and-in call (spot=80, barrier=85, K=100) vs vanilla call:

                   KI triggered        Vanilla           diff
-------------------------------------------------------------
present_value          2.710911       2.710911       0.000000
delta                  0.253543       0.253543       0.000000
gamma                  0.015852       0.015852       0.000000
vega                   0.253561       0.253561       0.000000
theta                 -0.009983      -0.009983       0.000000
rho                    0.175706       0.175706       0.000000


Every greek and PV match the vanilla call exactly:

- **PV** matches because each engine recognises the inception-triggered
  KI internally and returns the vanilla price directly.
- **NUMERICAL greeks** match because the OV-level short-circuit
  delegates the call to a memoised `_vanilla_equivalent_valuation`
  (a `VanillaSpec` clone of the parent KI spec), so both sides
  bump-and-revalue on the same vanilla and produce byte-identical
  numbers. 

## 6) Redundant Barriers — KI Specs Equivalent to Vanilla

Some KI specs have a barrier that is **automatically breached on any
in-the-money path**.  The classic examples:

- **UI call with $S_0 < H < K$** (barrier between spot and strike):
  the call pays $\max(S_T - K, 0)$, so for any positive payoff
  $S_T > K > H$.  By the intermediate value theorem on the continuous
  GBM path, $S$ must have passed through $H$ at some point — the UI
  barrier is automatically hit on every ITM path.  Conclusion: a UI
  call with $H \in (S_0, K)$ prices identically to a vanilla call.
- **DI put with $S_0 > H > K$** (barrier between strike and spot):
  symmetric.  $S_T < K$ requires $S$ passing through $H$, so the DI
  barrier is automatically hit on every ITM path.  DI put = vanilla
  put.

**The package has no short-circuit for these cases** — each engine
computes the barrier price honestly, and the result emerges as the
vanilla price by construction.  This is a useful sanity check that
the engines handle the limit correctly.  (The "redundant KO" mirror —
UO call with $H \in (S_0, K)$, DO put with $H \in (K, S_0)$ — is the
*opposite* limit: every ITM path is killed, so those redundant KOs
price as zero/rebate/discounted rebate.)

In [20]:
# ── Redundant UI call ────────────────────────────────────────────────
ud_redundant_ui = dp.UnderlyingData(
    initial_value=80.0,  # spot below barrier
    volatility=sigma,
    market_data=market_data,
    dividend_curve=curve_q,
)
spec_ui_call = dp.BarrierSpec(
    option_type=dp.OptionType.CALL,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=100.0,  # strike above barrier
    maturity=maturity,
    barrier=90.0,  # H ∈ (S, K) → redundant
    direction=dp.BarrierDirection.UP,
    action=dp.BarrierAction.IN,
    monitoring=dp.BarrierMonitoring.CONTINUOUS,
)
spec_vanilla_call_for_ui = dp.VanillaSpec(
    option_type=dp.OptionType.CALL,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=100.0,
    maturity=maturity,
)

# Sanity: the spec is NOT triggered at inception (spot 80 < barrier 90).
assert not spec_ui_call.is_spot_past_barrier(80.0), "expected UI call to be alive at inception"

# Build a GBM process for MC (the shared `gbm` is at S0=100; we need S=80 here).
gbm_redundant_ui = dp.GBMProcess(
    market_data=market_data,
    process_params=dp.GBMParams(initial_value=80.0, volatility=sigma, dividend_curve=curve_q),
    sim_config=sim_config,
)

ui_bsm_pv = dp.OptionValuation(ud_redundant_ui, spec_ui_call, dp.PricingMethod.BSM).present_value()
ui_binom_pv = dp.OptionValuation(
    ud_redundant_ui, spec_ui_call, dp.PricingMethod.BINOMIAL
).present_value()
ui_pde_pv = dp.OptionValuation(
    ud_redundant_ui, spec_ui_call, dp.PricingMethod.PDE_FD
).present_value()
ui_mc_pv = dp.OptionValuation(
    gbm_redundant_ui, spec_ui_call, dp.PricingMethod.MONTE_CARLO
).present_value()

van_bsm_pv = dp.OptionValuation(
    ud_redundant_ui, spec_vanilla_call_for_ui, dp.PricingMethod.BSM
).present_value()
van_binom_pv = dp.OptionValuation(
    ud_redundant_ui, spec_vanilla_call_for_ui, dp.PricingMethod.BINOMIAL
).present_value()
van_pde_pv = dp.OptionValuation(
    ud_redundant_ui,
    spec_vanilla_call_for_ui,
    dp.PricingMethod.PDE_FD,
    dp.PDEParams.for_barriers(monitoring=dp.BarrierMonitoring.CONTINUOUS),
).present_value()
van_mc_pv = dp.OptionValuation(
    gbm_redundant_ui, spec_vanilla_call_for_ui, dp.PricingMethod.MONTE_CARLO
).present_value()

print("── Redundant UI call (S=80, H=90, K=100) vs vanilla call ──")
print(f"{'Engine':<14} {'UI call':>12} {'Vanilla':>12} {'diff':>12}")
print("-" * 52)
for name, ui, van in [
    ("BSM", ui_bsm_pv, van_bsm_pv),
    ("Binomial", ui_binom_pv, van_binom_pv),
    ("PDE_FD", ui_pde_pv, van_pde_pv),
    ("Monte Carlo", ui_mc_pv, van_mc_pv),
]:
    print(f"{name:<14} {ui:>12.6f} {van:>12.6f} {ui - van:>+12.6f}")

── Redundant UI call (S=80, H=90, K=100) vs vanilla call ──
Engine              UI call      Vanilla         diff
----------------------------------------------------
BSM                2.710911     2.710911    -0.000000
Binomial           2.711413     2.707135    +0.004278
PDE_FD             2.710934     2.710934    +0.000000
Monte Carlo        2.715379     2.715379    +0.000000


Note for PDE_FD, we pass in `PDEParams.for_barriers` to the vanilla valuation, so the 
config matches the barrier valuation config. This means the diff is exactly zero. 
For Binomial, it's not as straightforward given the _effective_num_steps logic 
is internal to the engine (so the diff is not exactly zero).

In [21]:
# ── Redundant DI put ─────────────────────────────────────────────────
ud_redundant_di = dp.UnderlyingData(
    initial_value=120.0,  # spot above barrier
    volatility=sigma,
    market_data=market_data,
    dividend_curve=curve_q,
)
spec_di_put = dp.BarrierSpec(
    option_type=dp.OptionType.PUT,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=100.0,  # strike below barrier
    maturity=maturity,
    barrier=110.0,  # H ∈ (K, S) → redundant
    direction=dp.BarrierDirection.DOWN,
    action=dp.BarrierAction.IN,
    monitoring=dp.BarrierMonitoring.CONTINUOUS,
)
spec_vanilla_put_for_di = dp.VanillaSpec(
    option_type=dp.OptionType.PUT,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=100.0,
    maturity=maturity,
)

assert not spec_di_put.is_spot_past_barrier(120.0), "expected DI put to be alive at inception"

gbm_redundant_di = dp.GBMProcess(
    market_data=market_data,
    process_params=dp.GBMParams(initial_value=120.0, volatility=sigma, dividend_curve=curve_q),
    sim_config=sim_config,
)

di_bsm_pv = dp.OptionValuation(ud_redundant_di, spec_di_put, dp.PricingMethod.BSM).present_value()
di_binom_pv = dp.OptionValuation(
    ud_redundant_di, spec_di_put, dp.PricingMethod.BINOMIAL
).present_value()
di_pde_pv = dp.OptionValuation(
    ud_redundant_di, spec_di_put, dp.PricingMethod.PDE_FD
).present_value()
di_mc_pv = dp.OptionValuation(
    gbm_redundant_di, spec_di_put, dp.PricingMethod.MONTE_CARLO
).present_value()

vp_bsm_pv = dp.OptionValuation(
    ud_redundant_di, spec_vanilla_put_for_di, dp.PricingMethod.BSM
).present_value()
vp_binom_pv = dp.OptionValuation(
    ud_redundant_di, spec_vanilla_put_for_di, dp.PricingMethod.BINOMIAL
).present_value()
vp_pde_pv = dp.OptionValuation(
    ud_redundant_di,
    spec_vanilla_put_for_di,
    dp.PricingMethod.PDE_FD,
    dp.PDEParams.for_barriers(monitoring=dp.BarrierMonitoring.CONTINUOUS),
).present_value()
vp_mc_pv = dp.OptionValuation(
    gbm_redundant_di, spec_vanilla_put_for_di, dp.PricingMethod.MONTE_CARLO
).present_value()

print("\n── Redundant DI put (S=120, H=110, K=100) vs vanilla put ──")
print(f"{'Engine':<14} {'DI put':>12} {'Vanilla':>12} {'diff':>12}")
print("-" * 52)
for name, di, van in [
    ("BSM", di_bsm_pv, vp_bsm_pv),
    ("Binomial", di_binom_pv, vp_binom_pv),
    ("PDE_FD", di_pde_pv, vp_pde_pv),
    ("Monte Carlo", di_mc_pv, vp_mc_pv),
]:
    print(f"{name:<14} {di:>12.6f} {van:>12.6f} {di - van:>+12.6f}")


── Redundant DI put (S=120, H=110, K=100) vs vanilla put ──
Engine               DI put      Vanilla         diff
----------------------------------------------------
BSM                2.898198     2.898198    +0.000000
Binomial           2.899471     2.897774    +0.001697
PDE_FD             2.898155     2.898155    +0.000000
Monte Carlo        2.903952     2.903952    +0.000000


- **BSM, PDE_FD and Monte Carlo** match the vanilla price exactly
- **Binomial** diff is very close to but not exactly zero 
(because num_steps differs across the barrier/vanilla valuations).

No short-circuit fired anywhere — the engines computed the actual
barrier prices and they emerged equal to vanilla as the geometry
demands.  This is a useful regression check on the engines' barrier
logic.

## 7) Rebates — Worked Examples

Notebook 06a §2.4 defined the three rebate flavours; let's price them on
a concrete contract.  Setup: down-and-out call, spot=100, barrier=85,
strike=100, σ=0.25, r=0.05, q=0.02, T=1Y.  We'll add a $R = 5$ rebate
under each timing convention and price across all engines.

In [22]:
spec_doc_no_rebate = dp.BarrierSpec(
    option_type=dp.OptionType.CALL,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=K,
    maturity=maturity,
    barrier=H_DOWN,
    direction=dp.BarrierDirection.DOWN,
    action=dp.BarrierAction.OUT,
    monitoring=dp.BarrierMonitoring.CONTINUOUS,
)
spec_doc_at_hit = dc_replace(spec_doc_no_rebate, rebate=5.0, rebate_timing=dp.RebateTiming.AT_HIT)
spec_doc_at_expiry = dc_replace(
    spec_doc_no_rebate, rebate=5.0, rebate_timing=dp.RebateTiming.AT_EXPIRY
)

# KI rebate: paid at expiry only if the barrier is *never* touched.
# (KI rebates must use AT_EXPIRY — the package validates this.)
spec_dic_no_rebate = dc_replace(spec_doc_no_rebate, action=dp.BarrierAction.IN)
spec_dic_no_touch = dc_replace(
    spec_dic_no_rebate, rebate=5.0, rebate_timing=dp.RebateTiming.AT_EXPIRY
)


def _print_engine_row(label, spec, src):
    bsm_pv = dp.OptionValuation(
        src if not isinstance(src, dp.GBMProcess) else underlying, spec, dp.PricingMethod.BSM
    ).present_value()
    binom_pv = dp.OptionValuation(underlying, spec, dp.PricingMethod.BINOMIAL).present_value()
    pde_pv = dp.OptionValuation(underlying, spec, dp.PricingMethod.PDE_FD).present_value()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        mc_pv = dp.OptionValuation(gbm, spec, dp.PricingMethod.MONTE_CARLO).present_value()
    print(f"{label:<32} {bsm_pv:>10.6f} {binom_pv:>12.6f} {pde_pv:>12.6f} {mc_pv:>12.6f}")


print("Rebate variants — DO call (KO) and DI call (KI), barrier=85:")
print(f"{'Variant':<32} {'BSM':>10} {'Binomial':>12} {'PDE_FD':>12} {'MC':>12}")
print("-" * 82)
_print_engine_row("DO call, no rebate", spec_doc_no_rebate, underlying)
_print_engine_row("DO call, AT_HIT rebate=5", spec_doc_at_hit, underlying)
_print_engine_row("DO call, AT_EXPIRY rebate=5", spec_doc_at_expiry, underlying)
_print_engine_row("DI call, no rebate", spec_dic_no_rebate, underlying)
_print_engine_row("DI call, no-touch rebate=5", spec_dic_no_touch, underlying)

Rebate variants — DO call (KO) and DI call (KI), barrier=85:
Variant                                 BSM     Binomial       PDE_FD           MC
----------------------------------------------------------------------------------
DO call, no rebate                 9.923414     9.926217     9.923395     9.945108
DO call, AT_HIT rebate=5          12.460043    12.463335    12.460026    12.482305
DO call, AT_EXPIRY rebate=5       12.383861    12.387196    12.383843    12.406113
DI call, no rebate                 1.200348     1.199774     1.200267     1.195498
DI call, no-touch rebate=5         3.496048     3.494943     3.495966     3.490639


- **DO call AT_HIT vs AT_EXPIRY**: AT_HIT pays $R$ at the (random)
  hit time $\tau$; AT_EXPIRY pays $R$ at maturity $T$ if the barrier
  has been touched.  Both legs share the same hit indicator
  $\mathbb{1}_{\{\tau \le T\}}$, so per hit path the only difference
  is when the cashflow lands.  Under positive rates, $\text{df}(0,
  \tau) \ge \text{df}(0, T)$ with strict inequality whenever $\tau <
  T$, so the AT_HIT rebate leg is always at least as valuable as
  the AT_EXPIRY rebate leg — strictly so for any KO with positive hit
  probability. 
- **DI call no-touch rebate**: paid at expiry **if and only if** the
  barrier is never touched (i.e. the option never activated).  This
  adds value to the DI: the holder either gets the activated vanilla
  payoff *or* the no-touch rebate, never neither.

**In-out parity check.**  The generalised parity from Notebook 06a §2.2
states $V_{\text{vanilla}} + R \cdot \text{df}(0,T) = V_{\text{KI}} + V_{\text{KO}}$
when both legs have an at-expiry rebate of $R$.  Let's verify:

In [23]:
T = 1.0  # year fraction
df_T = underlying.discount_curve.df(T)
v_vanilla = dp.OptionValuation(underlying, spec_vanilla, dp.PricingMethod.BSM).present_value()
v_ko_at_expiry = dp.OptionValuation(
    underlying, spec_doc_at_expiry, dp.PricingMethod.BSM
).present_value()
v_ki_no_touch = dp.OptionValuation(
    underlying, spec_dic_no_touch, dp.PricingMethod.BSM
).present_value()

lhs = v_vanilla + 5.0 * df_T
rhs = v_ki_no_touch + v_ko_at_expiry
print(f"V_vanilla + R · df(0, T) = {lhs:.6f}")
print(f"V_KI (no-touch) + V_KO (AT_EXPIRY) = {rhs:.6f}")
print(f"Parity error            = {abs(lhs - rhs):.6f}")

V_vanilla + R · df(0, T) = 15.879909
V_KI (no-touch) + V_KO (AT_EXPIRY) = 15.879909
Parity error            = 0.000000


> **Production note.** Across every regime covered in this (and the 06a_barrier_options) notebooks — European and American exercise, continuous and discrete monitoring, barrier close to and far from spot — `PricingMethod.PDE_FD` is the single engine that handles every case at high accuracy (with the default parameters).
>
> For production pricing and Greeks, default to `PDE_FD`.
>
> - Use analytical `BSM` pricing when closed-form speed is required for European, continuously monitored contracts.
> - Use `Binomial` primarily as a cross-check.
> - Reserve `Monte Carlo` for products with additional path dependence beyond the barrier or for non-GBM dynamics.

## 8) Summary

| Topic | Recommendation |
|---|---|
| **DOP / UOC American** | PDE_FD primary, Binomial cross-check (a few % off without large `num_steps`).  MC LSM has structural downward bias; warn-emitted at `__init__`. |
| **Barrier within ~1% of spot** | PDE_FD or BSM analytical for PV.  Binomial degrades silently (Boyle-Lau cap binds → warning); near-barrier binomial greeks raise via the `_stencil_straddles_barrier` guard rather than return misleading values. |
| **Inception-triggered specs** | Each engine handles PV and engine-native greeks natively (same answer regardless of engine).  The OV-level short-circuit fires for NUMERICAL bump-and-reprice greeks, to avoid bumping spot across the trigger boundary. |
| **Redundant KI barriers** ($S < H < K$ for UI call, $S > H > K$ for DI put) | No short-circuit — engines naturally converge to the vanilla price.  A useful regression check on the barrier engines. |
| **Rebates** | Validate against the generalised in-out parity ($V_{\text{vanilla}} + R \cdot \text{df}(0,T) = V_{\text{KI}}^{\text{no-touch}} + V_{\text{KO}}^{\text{AT-EXPIRY}}$). |

**References** (in addition to notebook 06a's references):

- Boyle, P. P. and Tian, Y. (1998).  Tables 1, 4, and 6 of "An Explicit Finite
  Difference Approach to the Pricing of Barrier Options",
  *Applied Mathematical Finance* **5**, 17–43.